In [0]:

# ============================================================
# CELDA 1. AJUSTE DE DISTRIBUCIONES DE RETRASOS
# ============================================================

# Objetivo:
# Comparar las distribuciones exponencial, gamma y
# lognormal para describir la duración de los retrasos.
#
# Utilizamos únicamente retrasos positivos de 15
# minutos o más, relacionados con nuestro proyecto.
#
# Este análisis es estadístico y complementario
# a los modelos XGBoost.

from pyspark.sql import functions as F
from scipy import stats
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. CARGAR DATOS
# ------------------------------------------------------------

vuelos = spark.table(
    "workspace.gold_flights.flights_features"
)

# Seleccionamos retrasos de al menos 15 minutos.
# Excluimos valores nulos y utilizamos una muestra
# reproducible para no saturar Databricks.

retrasos = (
    vuelos
    .filter(
        F.col("dep_delay").isNotNull() &
        (F.col("dep_delay") >= 15)
    )
    .select("dep_delay")
    .sample(
        withReplacement=False,
        fraction=0.08,
        seed=42
    )
    .toPandas()
)

x = retrasos["dep_delay"].to_numpy(
    dtype=np.float64
)

print("Retrasos analizados:", len(x))
print("Retraso promedio:", round(np.mean(x), 2), "min")
print("Mediana:", round(np.median(x), 2), "min")
print("Máximo:", round(np.max(x), 2), "min")

# ------------------------------------------------------------
# 2. AJUSTAR LAS DISTRIBUCIONES
# ------------------------------------------------------------

# Ajustamos las distribuciones al mismo conjunto
# de retrasos y fijamos loc=0.
#
# Como seleccionamos retrasos >=15, conviene
# modelar el exceso sobre el umbral de 15 minutos.
# Así comparamos las duraciones adicionales.

exceso = x - 15 + 0.001

# Exponencial
params_exp = stats.expon.fit(
    exceso,
    floc=0
)

# Gamma
params_gamma = stats.gamma.fit(
    exceso,
    floc=0
)

# Lognormal
params_lognorm = stats.lognorm.fit(
    exceso,
    floc=0
)

# ------------------------------------------------------------
# 3. COMPARAR AIC Y BIC
# ------------------------------------------------------------

# Menores valores de AIC y BIC indican mejor
# equilibrio entre ajuste y complejidad del modelo.

def calcular_metricas(nombre, distribucion, parametros):
    log_likelihood = np.sum(
        distribucion.logpdf(exceso, *parametros)
    )

    k = len(parametros) - 1  # loc está fijado en cero
    n = len(exceso)

    aic = 2 * k - 2 * log_likelihood
    bic = k * np.log(n) - 2 * log_likelihood

    return {
        "Distribución": nombre,
        "AIC": aic,
        "BIC": bic
    }

resultados = pd.DataFrame([
    calcular_metricas(
        "Exponencial",
        stats.expon,
        params_exp
    ),
    calcular_metricas(
        "Gamma",
        stats.gamma,
        params_gamma
    ),
    calcular_metricas(
        "Lognormal",
        stats.lognorm,
        params_lognorm
    )
])

resultados = resultados.sort_values("AIC")

print("\nCOMPARACIÓN DE DISTRIBUCIONES")
display(resultados.round(2))

mejor_distribucion = resultados.iloc[0]["Distribución"]

print(
    "\nMejor distribución según AIC:",
    mejor_distribucion
)

# ------------------------------------------------------------
# 4. VISUALIZAR LOS AJUSTES
# ------------------------------------------------------------

# Mostramos el rango de 15 a 240 minutos
# para que los valores extremos no oculten
# la forma general de las distribuciones.

grid_exceso = np.linspace(
    0.001,
    225,
    500
)

plt.figure(figsize=(12, 6))

plt.hist(
    x,
    bins=60,
    range=(15, 240),
    density=True,
    alpha=0.4,
    label="Retrasos observados"
)

plt.plot(
    grid_exceso + 15,
    stats.expon.pdf(
        grid_exceso,
        *params_exp
    ),
    label="Exponencial"
)

plt.plot(
    grid_exceso + 15,
    stats.gamma.pdf(
        grid_exceso,
        *params_gamma
    ),
    label="Gamma"
)

plt.plot(
    grid_exceso + 15,
    stats.lognorm.pdf(
        grid_exceso,
        *params_lognorm
    ),
    label="Lognormal"
)

plt.title(
    "Ajuste de distribuciones a retrasos de 15 minutos o más"
)
plt.xlabel("Minutos de retraso")
plt.ylabel("Densidad")
plt.xlim(15, 240)
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print("\nANÁLISIS DE DISTRIBUCIONES COMPLETADO.")


### Statistical Distribution Analysis — Results

We analyzed a reproducible sample of 156,413 flights with departure delays of at least 15 minutes.

The average departure delay was 70.12 minutes, with a median of 43 minutes and a maximum observed delay of 2,521 minutes.

Three probability distributions were fitted to the delay duration exceeding the 15-minute threshold: exponential, gamma, and lognormal.

The gamma distribution obtained the lowest AIC (1,528,592.88) and BIC (1,528,612.80), outperforming the exponential and lognormal alternatives.

These results suggest that gamma provides the best relative fit among the evaluated distributions. However, further goodness-of-fit analysis would be necessary to determine how accurately it represents extreme delays.

This analysis complements the XGBoost classification models by describing the duration of significant departure delays rather than predicting whether an individual flight will be delayed.
